In [6]:
#importing libraries
import os
import zipfile
import shutil
import math
import pandas as pd
from google.colab import files

Step 1 : Upload files and define I/O folders

In [9]:
#upload zip file
uploaded = files.upload()

Saving Training video 1 - roboflow.zip to Training video 1 - roboflow.zip


In [10]:
zip_path = "/content/Training video 1 - roboflow.zip"  #path of the zip file

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [11]:
#define input and output folders
#input folders
image_folder = "/content/dataset/train/images"
label_folder = "/content/dataset/train/labels"

# Output folders
clean_image_folder = "clean_dataset/images"
clean_label_folder = "clean_dataset/labels"
rej_image_folder = "rejected_dataset/images"
rej_label_folder = "rejected_dataset/labels"

In [12]:
#make folders
os.makedirs(clean_image_folder, exist_ok=True)
os.makedirs(clean_label_folder, exist_ok=True)
os.makedirs(rej_image_folder, exist_ok=True)
os.makedirs(rej_label_folder, exist_ok=True)

Step 2 : Reject full-frame boxes (if any)

In [13]:
# Filtering threshold
big_bb = 0.95
accepted_count = 0
rejected_count = 0

In [14]:
# Filter images
for label_file in os.listdir(label_folder):
    if not label_file.endswith(".txt"):
        continue

    label_path = os.path.join(label_folder, label_file)
    image_name = label_file.replace(".txt", ".jpg")
    image_path = os.path.join(image_folder, image_name)

    if not os.path.exists(image_path):
        print(f"Missing image: {image_name}")
        continue

    reject = False
    with open(label_path, "r") as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, _, _, width, height = map(float, parts)
            if width >= big_bb or height >= big_bb:
                reject = True
                break

    if reject:
        shutil.copy(image_path, os.path.join(rej_image_folder, image_name))
        shutil.copy(label_path, os.path.join(rej_label_folder, label_file))
        rejected_count += 1
        print(f"Rejected: {image_name}")
    else:
        shutil.copy(image_path, os.path.join(clean_image_folder, image_name))
        shutil.copy(label_path, os.path.join(clean_label_folder, label_file))
        accepted_count += 1

Step 3 : Splitting into "with" and "no" boxes

In [15]:
#output folders
with_box_img = "with_boxes"
with_box_lbl = "with_boxes_labels"
no_box_img = "no_boxes"
no_box_lbl = "no_boxes_labels"

#create output folders
os.makedirs(with_box_img, exist_ok=True)
os.makedirs(with_box_lbl, exist_ok=True)
os.makedirs(no_box_img, exist_ok=True)
os.makedirs(no_box_lbl, exist_ok=True)

#counters
count_with_boxes = 0
count_no_boxes = 0

for label_file in os.listdir(label_folder):
    if not label_file.endswith(".txt"):
        continue

    label_path = os.path.join(label_folder, label_file)
    image_name = label_file.replace(".txt", ".jpg")
    image_path = os.path.join(image_folder, image_name)

    if not os.path.exists(image_path):
        print(f"Missing image: {image_name}")
        continue

    # Check if label file is empty
    if os.stat(label_path).st_size == 0:
        shutil.copy(image_path, os.path.join(no_box_img, image_name))
        shutil.copy(label_path, os.path.join(no_box_lbl, label_file))
        count_no_boxes += 1
    else:
        shutil.copy(image_path, os.path.join(with_box_img, image_name))
        shutil.copy(label_path, os.path.join(with_box_lbl, label_file))
        count_with_boxes += 1

# Final summary
print("\n Summary:")
print(f"Images with bounding boxes: {count_with_boxes}")
print(f"Images with NO bounding boxes: {count_no_boxes}")
print(f"Total processed: {count_with_boxes + count_no_boxes}")



 Summary:
Images with bounding boxes: 240
Images with NO bounding boxes: 1285
Total processed: 1525


Step 4 : Counting Microgels

In [16]:
microgel_output_folder = "microgel_appearances"
os.makedirs(microgel_output_folder, exist_ok=True)

seen_centers = [] #track of previously seen centers
new_microgel_count = 0  #count new unique microgels
frame_index = 0

label_files = sorted([f for f in os.listdir(with_box_lbl) if f.endswith(".txt")])

for label_file in label_files:
    label_path = os.path.join(with_box_lbl, label_file)
    frame_name = label_file.replace(".txt", ".jpg")
    frame_path = os.path.join(with_box_img, frame_name)

    with open(label_path, "r") as f:
        lines = f.readlines()

    found_new_in_frame = False
    DIST_THRESHOLD = 0.07 if frame_index < 100 else 0.05

    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        _, x_center, y_center, _, _ = map(float, parts)
        current_center = (x_center, y_center)

        is_new = True
        for prev_center in seen_centers:
            dx = prev_center[0] - current_center[0]
            dy = prev_center[1] - current_center[1]
            dist = math.sqrt(dx**2 + dy**2)

            if dist < DIST_THRESHOLD and current_center[0] <= prev_center[0]:
                is_new = False
                break

        if is_new:
            seen_centers.append(current_center)
            new_microgel_count += 1
            found_new_in_frame = True

    if found_new_in_frame and os.path.exists(frame_path):
        shutil.copy(frame_path, os.path.join(microgel_output_folder, frame_name))

    frame_index += 1

print(f"\n Total unique microgels estimated: {new_microgel_count}")
print(f" Total frames saved: {len(os.listdir(microgel_output_folder))}")


 Total unique microgels estimated: 30
 Total frames saved: 29


Step 5 : Downloading all folders

In [19]:
folders_to_download = [
    'clean_dataset', 'rejected_dataset', 'with_boxes', 'with_boxes_labels',
    'no_boxes', 'no_boxes_labels', 'microgel_appearances'
]

for folder_name in folders_to_download:
    zip_name = f"{folder_name}.zip"

    shutil.make_archive(folder_name, 'zip', folder_name)
    print(f"Zipped: {zip_name}")

    files.download(zip_name)


Zipped: clean_dataset.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Zipped: rejected_dataset.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (11 outputs left out)

<IPython.core.display.Javascript object>

Zipped: microgel_appearances.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>